# Binär ↔ Dezimal – Interaktive Übungen (1. Lehrjahr)

**Ziele**
- Binärzahl in Dezimalzahl umrechnen (z. B. `110101₂ → 53₁₀`)
- Dezimalzahl in Binärzahl zerlegen (z. B. `53₁₀ → 110101₂`)

**So arbeitest du**
1. Starte bei **3.1 Binär → Dezimal**. Klicke die Kästchen an und beobachte, wie sich die Summe ändert.
2. Wechsle zu **3.2 Dezimal → Binär**. Gib eine Zahl ein und sieh die Zerlegung.
3. Übe mit den **Quiz**‑Aufgaben.

> Hinweis: Dieses Notebook funktioniert in Google Colab. Falls Widgets nicht reagieren, führe oben **Laufzeit → Alle ausführen** aus.

Bitte die nächsten zwei Downloads ausführen bevor du mit den Übungen startest


In [ ]:
#@title Download starten { display-mode: "form" }
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://raw.githubusercontent.com/IneichenEdulu/Mathe_Notebook/main/bin_dec_helpers.py')

Downloaded bin_dec_helpers.py


'bin_dec_helpers.py'

In [ ]:
#@title Import starten { display-mode: "form" }
# Imports (Module + Widgets)
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import random

from bin_dec_helpers import (
    BitSystem, DEFAULT_SYSTEM,
    bits_to_decimal, decimal_to_bits, format_equation, greedy_steps,
    sanitize_binary_string, bits_from_string, bits_to_string, bit_columns
)


## Hintergrund – was passiert hier eigentlich?

Wir arbeiten mit **6 Bit** (Werte `32, 16, 8, 4, 2, 1`).

### 3.1 Binär → Dezimal (Beispiel „110101“)
- Linke Ziffer `1` bedeutet **32 Punkte**.  
- Nächste `1` bedeutet **16 Punkte**.  
- Dann `0` → **0 Punkte**.  
- Dann `1` → **4 Punkte**.  
- Dann `0` → **0 Punkte**.  
- Letzte `1` → **1 Punkt**.  
**Summe:** `32 + 16 + 0 + 4 + 0 + 1 = 53`.  
Also: `110101₂ = 53₁₀`.

### 3.2 Dezimal → Binär (Beispiel „53“)
Wir prüfen der Reihe nach: `32, 16, 8, 4, 2, 1`.
1. Passt `32` in `53`? **Ja** → `1`, Rest `21`.  
2. Passt `16` in `21`? **Ja** → `1`, Rest `5`.  
3. Passt `8` in `5`? **Nein** → `0`.  
4. Passt `4` in `5`? **Ja** → `1`, Rest `1`.  
5. Passt `2` in `1`? **Nein** → `0`.  
6. Passt `1` in `1`? **Ja** → `1`, Rest `0`.  
Ergebnis: `110101₂`.

### 3.1 Binär → Dezimal

In [ ]:
#@title Binär → Dezimal (interaktiv) starten { display-mode: "form" }
# 3.1 Binär → Dezimal (interaktiv)
def build_binary_to_decimal_widget(system: BitSystem) -> widgets.VBox:
    weights = system.weights
    checkboxes = [widgets.Checkbox(value=False, indent=False, layout=widgets.Layout(width='25px', height='25px')) for _ in weights]
    for cb in checkboxes:
        cb.style.description_width = '0'
    binary_text = widgets.Text(value='', placeholder='z. B. 110101', description='Binär:')
    show_zeros_toggle = widgets.Checkbox(value=True, description='0‑Beiträge anzeigen')
    example_btn = widgets.Button(description='Beispiel 110101 laden')
    random_btn = widgets.Button(description='Zufällige Bits')
    reset_btn = widgets.Button(description='Alles 0')
    output_html = widgets.HTML(value="")
    status_html = widgets.HTML(value="")
    cols = bit_columns(weights, checkboxes)

    updating = {'flag': False}

    def update_output():
        if updating['flag']:
            return
        updating['flag'] = True
        bits = [1 if cb.value else 0 for cb in checkboxes]
        equation = format_equation(bits, weights, show_zeros_toggle.value)
        decimal_value = bits_to_decimal(bits, weights)
        bin_str = bits_to_string(bits)
        output_html.value = (
            f"<h4>Ergebnis</h4>"
            f"<div>Binär: <code>{bin_str}</code> → Dezimal: <b>{decimal_value}</b></div>"
            f"<div>Summe: <code>{equation}</code></div>"
        )
        binary_text.value = bin_str
        status_html.value = ""
        updating['flag'] = False

    def on_text_change(change):
        if updating['flag']:
            return
        updating['flag'] = True
        raw = change['new']
        cleaned = sanitize_binary_string(raw)
        if cleaned != raw:
            status_html.value = "<span style='color:#b00'>Hinweis: Nur 0/1 sind erlaubt. Andere Zeichen wurden entfernt.</span>"
        bits = bits_from_string(cleaned, width=len(weights))
        for cb, b in zip(checkboxes, bits):
            cb.value = bool(b)
        updating['flag'] = False
        update_output()

    for cb in checkboxes:
        cb.observe(lambda change: update_output(), names='value')
    show_zeros_toggle.observe(lambda change: update_output(), names='value')
    binary_text.observe(on_text_change, names='value')

    def on_example_click(_):
        s = "110101"
        bits = bits_from_string(s, width=len(weights))
        for cb, b in zip(checkboxes, bits):
            cb.value = bool(b)
        update_output()

    def on_random_click(_):
        bits = [random.randint(0, 1) for _ in weights]
        for cb, b in zip(checkboxes, bits):
            cb.value = bool(b)
        update_output()

    def on_reset_click(_):
        for cb in checkboxes:
            cb.value = False
        update_output()

    example_btn.on_click(on_example_click)
    random_btn.on_click(on_random_click)
    reset_btn.on_click(on_reset_click)

    on_example_click(None)

    buttons = widgets.HBox([example_btn, random_btn, reset_btn, show_zeros_toggle],
                           layout=widgets.Layout(gap='8px', flex_flow='row wrap'))
    return widgets.VBox([
        widgets.HTML("<h3>3.1 Binär → Dezimal</h3><p>Aktiviere (1) oder deaktiviere (0) die Kästchen. Jede Spalte steht für einen Wert.</p>"),
        cols,
        widgets.HBox([binary_text], layout=widgets.Layout(justify_content='center')),
        buttons,
        output_html,
        status_html
    ])

ui_31 = build_binary_to_decimal_widget(DEFAULT_SYSTEM)
display(ui_31)


### 3.2 Dezimal → Binär

In [ ]:
#@title Dezimal → Binär (interaktiv) starten { display-mode: "form" }
# 3.2 Dezimal → Binär (interaktiv)
def build_decimal_to_binary_widget(system: BitSystem) -> widgets.VBox:
    weights = system.weights
    max_val = sum(weights)
    dec_input = widgets.BoundedIntText(value=53, min=0, max=max_val, step=1, description='Dezimal:')
    example_btn = widgets.Button(description='Beispiel 53 laden')
    random_btn = widgets.Button(description='Zufallszahl')
    steps_html = widgets.HTML()
    output_summary = widgets.HTML()

    checkboxes = [widgets.Checkbox(value=False, disabled=True, indent=False, layout=widgets.Layout(width='25px', height='25px')) for _ in weights]
    cols = bit_columns(weights, checkboxes)

    def update_view(*_):
        n = dec_input.value
        bits = decimal_to_bits(n, weights)
        for cb, b in zip(checkboxes, bits):
            cb.value = bool(b)
        equation = format_equation(bits, weights, show_zeros=True)
        bin_str = bits_to_string(bits)
        output_summary.value = (
            f"<h4>Ergebnis</h4>"
            f"<div>Dezimal: <b>{n}</b> → Binär: <code>{bin_str}</code></div>"
            f"<div>Summe: <code>{equation}</code></div>"
        )
        steps_html.value = "<h4>Greedy‑Schritte</h4>" + greedy_steps(n, weights)

    dec_input.observe(update_view, names='value')

    def on_example(_):
        dec_input.value = 53

    def on_random(_):
        dec_input.value = random.randint(0, max_val)

    example_btn.on_click(on_example)
    random_btn.on_click(on_random)

    update_view()

    buttons = widgets.HBox([example_btn, random_btn], layout=widgets.Layout(gap='8px'))
    header = widgets.HBox([dec_input, buttons], layout=widgets.Layout(gap='8px', align_items='center'))
    return widgets.VBox([
        widgets.HTML("<h3>3.2 Dezimal → Binär</h3><p>Gib eine Zahl ein. Das Notebook füllt die Kästchen – genau wie im Beispiel.</p>"),
        header,
        cols,
        output_summary,
        steps_html
    ])

ui_32 = build_decimal_to_binary_widget(DEFAULT_SYSTEM)
display(ui_32)


### Übungen

In [ ]:
#@title Übungen starten { display-mode: "form" }
# Übungen
def build_quiz_binary_to_decimal(system: BitSystem) -> widgets.VBox:
    weights = system.weights
    state = {'bits': [1,1,0,1,0,1]}
    task_html = widgets.HTML()
    answer = widgets.BoundedIntText(value=0, min=0, max=sum(weights), description='Antwort:')
    new_btn = widgets.Button(description='Neue Aufgabe')
    check_btn = widgets.Button(description='Prüfen')
    feedback = widgets.HTML()

    def new_task(_=None):
        import random as _rnd
        state['bits'] = [_rnd.randint(0,1) for _ in weights]
        s = bits_to_string(state['bits'])
        task_html.value = f"<h4>Aufgabe</h4>Wandle die Binärzahl <code>{s}</code> in Dezimal um."
        answer.value = 0
        feedback.value = ""

    def check(_):
        expected = bits_to_decimal(state['bits'], weights)
        if answer.value == expected:
            feedback.value = "<b style='color:green'>Richtig! ✅</b>"
        else:
            eq = format_equation(state['bits'], weights, show_zeros=True)
            feedback.value = f"<b style='color:#b00'>Noch nicht. ❌</b> Richtige Lösung: <code>{eq}</code>."

    new_btn.on_click(new_task)
    check_btn.on_click(check)
    new_task()

    actions = widgets.HBox([new_btn, check_btn], layout=widgets.Layout(gap='8px'))
    return widgets.VBox([widgets.HTML("<h3>Quiz: Binär → Dezimal</h3>"), task_html, answer, actions, feedback])

def build_quiz_decimal_to_binary(system: BitSystem) -> widgets.VBox:
    weights = system.weights
    target = {'n': 0}
    task_html = widgets.HTML()
    check_btn = widgets.Button(description='Prüfen')
    new_btn = widgets.Button(description='Neue Aufgabe')
    feedback = widgets.HTML()
    checkboxes = [widgets.Checkbox(value=False, indent=False, layout=widgets.Layout(width='25px', height='25px')) for _ in weights]
    cols = bit_columns(weights, checkboxes)

    def new_task(_=None):
        import random as _rnd
        target['n'] = _rnd.randint(0, sum(weights))
        task_html.value = f"<h4>Aufgabe</h4>Stelle die Dezimalzahl <b>{target['n']}</b> als Binärzahl dar, indem du die Kästchen setzt."
        for cb in checkboxes:
            cb.value = False
        feedback.value = ""

    def check(_):
        bits = [1 if cb.value else 0 for cb in checkboxes]
        val = bits_to_decimal(bits, weights)
        expected_bits = decimal_to_bits(target['n'], weights)
        expected_str = bits_to_string(expected_bits)
        given_str = bits_to_string(bits)
        if val == target['n']:
            feedback.value = f"<b style='color:green'>Richtig! ✅</b> Deine Bits: <code>{given_str}</code>"
        else:
            eq = format_equation(expected_bits, weights, show_zeros=True)
            feedback.value = f"<b style='color:#b00'>Noch nicht. ❌</b> Richtige Bits: <code>{expected_str}</code> (Summe: <code>{eq}</code>)"

    check_btn.on_click(check)
    new_btn.on_click(new_task)
    new_task()

    actions = widgets.HBox([new_btn, check_btn], layout=widgets.Layout(gap='8px'))
    return widgets.VBox([widgets.HTML("<h3>Quiz: Dezimal → Binär</h3>"), task_html, cols, actions, feedback])

quiz_b2d = build_quiz_binary_to_decimal(DEFAULT_SYSTEM)
quiz_d2b = build_quiz_decimal_to_binary(DEFAULT_SYSTEM)

accordion = widgets.Accordion(children=[quiz_b2d, quiz_d2b])
accordion.set_title(0, "Übungen: Binär → Dezimal")
accordion.set_title(1, "Übungen: Dezimal → Binär")
display(accordion)


Accordion(children=(VBox(children=(HTML(value='<h3>Quiz: Binär → Dezimal</h3>'), HTML(value='<h4>Aufgabe</h4>W…

## 5 Addition und Multiplikation mit Binärzahlen (interaktiv)

**Regeln (kurz):**
- Addition: `0+0=0`, `1+0=0+1=1`, `1+1=10` (0 schreiben, 1 übertragen)
- Multiplikation: `0×0=0`, `1×1=1`; bei jeder `1` im Multiplikator wird der Multiplikand um die Stellenzahl nach links verschoben und addiert.


In [ ]:
#@title Binär-Addition und -Multiplikation starten { display-mode: "form" }
# Interaktive Aufgaben: Binär-Addition und -Multiplikation
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Wir verwenden die vorhandenen Helfer aus bin_dec_helpers.py
from bin_dec_helpers import (
    BitSystem, DEFAULT_SYSTEM,
    sanitize_binary_string, bits_to_string
)

def _sanitize(s: str) -> str:
    '''Bereinigt eine Binär-Eingabe: entfernt Leerzeichen/Unterstriche, validiert 0/1.'''
    s2 = sanitize_binary_string(s)
    if s2 is None:
        raise ValueError('Nur Ziffern 0/1 erlaubt.')
    # führende Nullen entfernen (aber mind. eine Ziffer behalten)
    s2 = s2.lstrip('0') or '0'
    return s2

def _add_bin(a: str, b: str) -> str:
    return format(int(a, 2) + int(b, 2), 'b')

def _mul_bin(a: str, b: str) -> str:
    return format(int(a, 2) * int(b, 2), 'b')

def _addition_steps_html(a: str, b: str) -> str:
    '''Erstellt eine einfache Spalten-Rechnung mit Überträgen als HTML (<pre>).'''
    # rechtsbündig ausrichten
    w = max(len(a), len(b))
    A = a.zfill(w)
    B = b.zfill(w)
    carry = 0
    result = []
    carries = []
    for i in range(w-1, -1, -1):
        ai = int(A[i])
        bi = int(B[i])
        s = ai + bi + carry
        result.append(str(s & 1))
        carries.append(carry)
        carry = 1 if s >= 2 else 0
    result.reverse()
    carries.reverse()
    res = ''.join(result)
    if carry:
        res = '1' + res

    # hübsches Layout
    top = '  ' + A
    mid = '+ ' + B
    sep = ' ' + '-' * (max(len(top), len(mid))-1)
    bot = '= ' + res.rjust(max(len(A), len(B)), ' ')
    # Überträge eine Zeile darüber (letzter Übertrag ganz links)
    carry_line = []
    next_carry = 0
    for c in carries:
        carry_line.append('^' if c else ' ')
    if carry:
        carry_line = ['^'] + carry_line
    carry_str = ' ' + ''.join(carry_line).rjust(len(sep)-1, ' ')
    return f'<pre>{carry_str}\n{top}\n{mid}\n{sep}\n{bot}</pre>'

def _partial_products_html(a: str, b: str) -> str:
    '''Zeigt partielle Produkte für a * b (b ist Multiplikator, von rechts).'''
    a, b = a.lstrip('0') or '0', b.lstrip('0') or '0'
    parts = []
    for i, bit in enumerate(reversed(b)):
        if bit == '1':
            parts.append((i, a + '0'*i))
        else:
            parts.append((i, '0' if a=='0' else '0'* (len(a)+i)))
    # Ergebnis
    res = _mul_bin(a,b)
    width = max([len(p)+2 for _, p in parts] + [len(a), len(b), len(res)])

    lines = []
    lines.append('  ' + a.rjust(width-2))
    lines.append('× ' + b.rjust(width-2))
    lines.append(' ' + '-'*(width-1))
    for i,(shift, p) in enumerate(parts):
        prefix = '  '
        if i == len(parts)-1:
            prefix = '  '  # same
        if set(p) == {'0'}:
            p = '0'
        lines.append(prefix + p.rjust(width-2))
    if len(parts) > 1:
        lines.append(' ' + '-'*(width-1))
    lines.append('= ' + res.rjust(width-2))
    return '<pre>' + '\n'.join(lines) + '</pre>'

# Beide Operanden dürfen unabhängig vom angezeigten Stellenwertsystem bis zu 8 Bit haben.
MAX_OPERAND_BITS = 8

def build_binary_addition_widget(system: BitSystem) -> widgets.VBox:
    max_bits = MAX_OPERAND_BITS
    state = {'a': '1101', 'b': '101'}

    task_html = widgets.HTML()
    ans = widgets.Text(value='', placeholder='z. B. 1110110', description='Antwort:')
    check_btn = widgets.Button(description='Prüfen')
    new_btn = widgets.Button(description='Neue Aufgabe')
    show_solution = widgets.Checkbox(value=False, description='Lösung/Schritte anzeigen')
    feedback = widgets.HTML()

    def new_task(_=None):
        import random as _rnd
        la = _rnd.randint(2, max_bits)
        lb = _rnd.randint(2, max_bits)
        # Die erste Ziffer ist 1, damit die gewählte Bitlänge erhalten bleibt.
        a = '1' + ''.join(str(_rnd.randint(0,1)) for _ in range(la - 1))
        b = '1' + ''.join(str(_rnd.randint(0,1)) for _ in range(lb - 1))
        state['a'], state['b'] = a, b
        task_html.value = f'<h4>Aufgabe</h4>Addiere: <code>{a}₂ + {b}₂</code>'
        ans.value = ''
        feedback.value = ''

    def check(_=None):
        try:
            given = _sanitize(ans.value)
        except Exception as e:
            feedback.value = f'<b style="color:#b00">Eingabe ungültig:</b> {e}'
            return
        a, b = state['a'], state['b']
        exp = _add_bin(a, b)
        if given == exp:
            hint = f' (dezimal: {int(a,2)} + {int(b,2)} = {int(exp,2)})'
            extra = _addition_steps_html(a,b) if show_solution.value else ''
            feedback.value = f'<b style="color:green">Richtig! ✅</b>{hint}{extra}'
        else:
            extra = ''
            if show_solution.value:
                extra = _addition_steps_html(a,b) + f'<div>Erwartet: <code>{exp}₂</code></div>'
            else:
                # einfacher Tipp: erste fehlerhafte Spalte von rechts finden
                def first_diff(a,b,given):
                    r = _add_bin(a,b)
                    # rechtsbündig ausrichten
                    w = max(len(r), len(given))
                    R = r.zfill(w); G = given.zfill(w)
                    for i in range(1, w+1):
                        if R[-i] != G[-i]:
                            return i  # i-te Stelle von rechts
                    return None
                k = first_diff(a,b,given)
                tipp = f' Tipp: Prüfe Stelle {k} von rechts.' if k else ''
                extra = tipp
            feedback.value = f'<b style="color:#b00">Noch nicht richtig. ❌</b>{extra}'

    new_btn.on_click(new_task)
    check_btn.on_click(check)
    new_task()

    controls = widgets.HBox([new_btn, check_btn, show_solution], layout=widgets.Layout(gap='8px'))
    return widgets.VBox([widgets.HTML('<h3>Addition (Binär)</h3>'), task_html, ans, controls, feedback])


def build_binary_multiplication_widget(system: BitSystem) -> widgets.VBox:
    max_bits = MAX_OPERAND_BITS
    state = {'a': '1011', 'b': '101'}
    task_html = widgets.HTML()
    ans = widgets.Text(value='', placeholder='z. B. 110111', description='Antwort:')
    check_btn = widgets.Button(description='Prüfen')
    new_btn = widgets.Button(description='Neue Aufgabe')
    show_solution = widgets.Checkbox(value=False, description='Lösung/Teilsummen anzeigen')
    feedback = widgets.HTML()

    def new_task(_=None):
        import random as _rnd
        la = _rnd.randint(2, max_bits)
        lb = _rnd.randint(2, max_bits)
        # Die erste Ziffer ist 1, damit die gewählte Bitlänge erhalten bleibt.
        a = '1' + ''.join(str(_rnd.randint(0,1)) for _ in range(la - 1))
        b = '1' + ''.join(str(_rnd.randint(0,1)) for _ in range(lb - 1))
        state['a'], state['b'] = a, b
        task_html.value = f'<h4>Aufgabe</h4>Multipliziere: <code>{a}₂ × {b}₂</code>'
        ans.value = ''
        feedback.value = ''

    def check(_=None):
        try:
            given = _sanitize(ans.value)
        except Exception as e:
            feedback.value = f'<b style="color:#b00">Eingabe ungültig:</b> {e}'
            return
        a, b = state['a'], state['b']
        exp = _mul_bin(a, b)
        if given == exp:
            hint = f' (dezimal: {int(a,2)} × {int(b,2)} = {int(exp,2)})'
            extra = _partial_products_html(a,b) if show_solution.value else ''
            feedback.value = f'<b style="color:green">Richtig! ✅</b>{hint}{extra}'
        else:
            extra = _partial_products_html(a,b) + f'<div>Erwartet: <code>{exp}₂</code></div>' if show_solution.value else ''
            feedback.value = f'<b style="color:#b00">Noch nicht richtig. ❌</b>{extra}'

    new_btn.on_click(new_task)
    check_btn.on_click(check)
    new_task()

    controls = widgets.HBox([new_btn, check_btn, show_solution], layout=widgets.Layout(gap='8px'))
    return widgets.VBox([widgets.HTML('<h3>Multiplikation (Binär)</h3>'), task_html, ans, controls, feedback])


# Anzeigen
w_add = build_binary_addition_widget(DEFAULT_SYSTEM)
w_mul = build_binary_multiplication_widget(DEFAULT_SYSTEM)

acc = widgets.Accordion(children=[w_add, w_mul])
acc.set_title(0, 'Addition (Binär) – interaktiv')
acc.set_title(1, 'Multiplikation (Binär) – interaktiv')
display(acc)

Accordion(children=(VBox(children=(HTML(value='<h3>Addition (Binär)</h3>'), HTML(value='<h4>Aufgabe</h4>Addier…

In [ ]:
#@title Aufgaben aus dem Skript { display-mode: "form" }
# Feste Aufgaben aus dem Skript
# Aufgabe 5: 1101101₂ + 1001₂
# Aufgabe 6: 1011101₂ × 111₂

import ipywidgets as widgets
from IPython.display import display, HTML
from bin_dec_helpers import sanitize_binary_string

def _sanitize(s: str) -> str:
    s2 = sanitize_binary_string(s)
    if s2 is None:
        raise ValueError('Nur Ziffern 0/1 erlaubt.')
    return s2.lstrip('0') or '0'

def _add_bin(a: str, b: str) -> str:
    return format(int(a, 2) + int(b, 2), 'b')

def _mul_bin(a: str, b: str) -> str:
    return format(int(a, 2) * int(b, 2), 'b')

def _row(title, a, b, kind):
    if kind == 'add':
        exp = _add_bin(a,b)
        op = ' + '
    else:
        exp = _mul_bin(a,b)
        op = ' × '
    task = widgets.HTML(value=f'<b>{title}</b>: <code>{a}₂{op}{b}₂</code>')
    ans = widgets.Text(placeholder='Ihre Lösung in Binär', description='Antwort:')
    fb  = widgets.HTML(value='')
    btn = widgets.Button(description='Prüfen')

    def check(_=None):
        try:
            given = _sanitize(ans.value)
        except Exception as e:
            fb.value = f'<b style="color:#b00">Eingabe ungültig:</b> {e}'
            return
        if given == exp:
            fb.value = f'<b style="color:green">Richtig! ✅</b> Erwartet: <code>{exp}₂</code>'
        else:
            fb.value = f'<b style="color:#b00">Nicht korrekt. ❌</b> Erwartet: <code>{exp}₂</code>'
    btn.on_click(check)
    return widgets.VBox([task, ans, btn, fb])

r1 = _row('Aufgabe 5', '1101101', '1001', 'add')
r2 = _row('Aufgabe 6', '1011101', '111', 'mul')

box = widgets.VBox([widgets.HTML('<h3>Feste Aufgaben (aus dem Skript)</h3>'), r1, r2])
display(box)


Copyright 2025 [Markus Ineichen](mailto:markus.ineichen1@sluz.ch)

Code licence: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)